# Session 5: Introduction to non-linear regression

In [ ]:
# As usual, we import the libraries we will need during the coding session

import numpy as np
import matplotlib.pyplot as plt

## Exercise 1: From linear to non-linear

In linear regression, the target function $y$ is represented by a **linear** combination of features $x$ such that $y = f(x) + \epsilon$, where $f(x) = \mathbf{w}^T \mathbf{x} = \sum_{i}^d w_i x_i$.

We need to tune the weights $w$ in order to minimise the residuals $\epsilon$ (i.e. the approximation error). For that, we first need to collect some data on the inputs $x$ and the target $y$. 

Then, in the Ordinary Least-Squares approach, the objective is to tune the weights to minimise the euclidean distance between the observations $y$ and the predictions $Xw$:

$w^*= \arg \underset{w}{\min} ||Xw - y||^2_2$

We can use this framework to determine the spring constant of a spring based on noisy data collected in the laboratory (where $x$ is the longitudinal force applied to the spring and $y$ is its length):

In [ ]:
# Fix the seed for reproducibility
np.random.seed(42)

# We collected n=20 measures
n = 20
x = np.linspace(0, 10, n)

# We already know the spring constant and the initial length given in its datasheet
k = 2
l0 = 10

# We made n noisy observations
msrmt_error = 1
y_obs = l0 + k * x + np.random.normal(0, msrmt_error, size=x.shape)

# Plot the data
plt.scatter(x, y_obs, color='k')

# Then, we determine the Hooke's law parameters by performing a linear regression on the data
from sklearn.linear_model import LinearRegression
model = ...
... # Pay attention to the shape of the data
y_pred = ...
plt.plot(x, y_pred, color='r')
print(f'K = {model.coef_[0]:.2f}')
print(f'l0 = {model.intercept_:.2f}')
print(f'R2 = {model.score(x.reshape(-1, 1), y_obs):.2f}')

# The R2 is very high! But what about the estimation of K? 

Now, let's illustrate the limitations: imagine that we are measuring the impulse response of a second-order damped system in automation laboratory. We know that our function must take the following form:

$y = A e^{-\alpha t} \sin(\omega t + \phi)$

where $y$ is the response and $t$ the time. $A$, $\alpha$, $\omega$, $\phi$ are our coefficients to determine. Note that these parameters are generally not part of a linear term.

In [ ]:
# Fix the seed for reproducibility
np.random.seed(42)

# We collected n=100 measures
n = 100
x = np.linspace(0, 10, n) # 10 seconds experiment

# Let's imagine we know the coefficients
A = 10
alpha = 0.5
omega = 2
phi = 0.5

# We made n noisy observations
msrmt_error = 0.5
y_obs = A * np.sin(omega * x + phi) * np.exp(-alpha * x) + np.random.normal(0, msrmt_error, size=x.shape)

# Plot the data
plt.scatter(x, y_obs, color='k')
plt.show()

How do we find the coefficients? The challenge is that they now appear inside nonlinear terms (sine, exponential...). One might be tempted to apply linear regression anyway ... try building a good feature matrix that suits our problem !

In [ ]:
# Try building a good feature matrix that suits our problem !

X = np.column_stack((...))

# Let's copy paste the previous code snippet to see ...

# Did you succeed ?

Linear regression simply is not the right tool when model parameters appear inside nonlinear terms. This is exactly the problem that nonlinear regression and gradient descent are designed to solve (see below).

## Exercise 2: Gradient descent

In linear regression, the least squares lost function can be expressed as:

$J = ||Xw - y||^2_2 = (Xw - y)^T(Xw -y) = (w^TX^TXw - 2w^TX^Ty + y^Ty)$

Minimising this cost function leads to setting its gradient with respect to $w$ to zero:

$\nabla_w J = 2X^TXw - 2X^Ty = 0$

This is a linear system with respect to $w$ and it can be easily solved:

$w^* = (X^TX)^{-1}X^Ty$

> Question: What would be the impact of adding Ridge's loss term: $\alpha ||w||_2^2$ on $w^*$ ?

For most non-linear models of practical interest, no general closed-form solution exists:

$J = ||f(X;w) - y||^2_2$


Then, setting the gradient of the loss to zero leads to a system of nonlinear equations. So instead, the idea is to search for the best values iteratively, starting from an initial guess, and progressively reducing the prediction error. This is what gradient descent, a widely used method in optimisation, aims to do.

### Main idea

Consider a general loss function $\mathcal{L}(\theta)$ that we want to minimise. If we know its gradient $\nabla_\theta \mathcal{L}$, we know locally which direction makes it increase. As we want to minimise $\mathcal{L}(\theta)$ we step in the opposite direction of that gradient:

$\theta_{k+1} =  \theta_k - \eta \, \nabla_\theta \mathcal{L}$

- $\nabla_\theta \mathcal{L}$ points uphill — we go the other way
- $\eta$ is the **learning rate** - the size of each step

Repeat this update iteratively, and $\theta$ will progressively converge toward a minimum of $\mathcal{L}$.

Let's draw an example below with $\mathcal{L} = 20(sin(\theta_1) \cdot cos(\theta_2)) + \theta_1^2 + \theta_2^2$ :

In [ ]:
# Consider x = theta1, y = theta2 and z = L for the sake of simplicity

x = np.linspace(-5, 5, 1000)
y = np.linspace(-5, 5, 1000)

X, Y = np.meshgrid(x, y)

Z = 20*(np.sin(X) * np.cos(Y)) + X**2 + Y**2

# Create Plots
fig= plt.figure(figsize=(10,5))

# Subplot 1: 3D Parametric Plot
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.grid()
surf = ax.plot_surface(X, Y, Z, cmap=plt.get_cmap('viridis'),alpha= 0.5 )
ax.set_title('3D Parametric Plot')

ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)
ax.set_zlabel('f', labelpad=20)


# Subplot 2: 2D Contour Plot
ax = fig.add_subplot(1, 2, 2)
# continuous contour plot
surf = ax.contourf(X, Y, Z, cmap=plt.get_cmap('viridis') )
ax.set_title('2D Contour Plot')

fig.colorbar(surf, shrink=0.5, aspect=8)

# Set axes label
ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)
plt.show()

### Gradient descent

We will write the gradient descent in form of a function with the following properties:

Input:
- The starting point (coordinates in x and y): `start_indepvar`
- The learning rate : `learning_rate`
- The number of iterations : `num_iterations`

Output:
- The end point (coordinates in x and y): `indepvar`
- The minimum function value after optimization : `z`
- The history of (x, y, z) over the iterations : `history`

Steps:
- Calculate the function value $\boldsymbol{z}_k = f(\boldsymbol{x}_k)$
- Calculate next point as $\boldsymbol{x}_{k+1} = \boldsymbol{x}_k - \delta \cdot \nabla f(\boldsymbol{x}_k)$ (gradient descent step)
- Check convergence :  $|f(\boldsymbol{x}_{k+1} - f(\boldsymbol{x}_k)|< \epsilon$
- Save the solution : `history`

Variables:
- Learning rate: $\delta$
- Threshold to stop the algorithm if convergence is reached: $\epsilon$
- Maximum number of iterations of the gradient descent algorithm: `num_iterations`
- Direction to locally decrease the function value: $\nabla f(\boldsymbol{x}_k)$

As you can observe in the steps above, we have to evaluate the function and compute its gradient at every iteration. Therefore, it is easier if we implement some help functions before to do these two tasks.

### Exercise 2: Step 1 - Build the help functions

Recall: $f(x, y) = 20 (sin(x) \cdot cos(y)) + x^2 + y^2$

#### function`f(indepvar)`:
- **description**: Evaluates the function f at coordinates (x, y)
- **input**: `indepvar` = array of size (2,): $\boldsymbol{x} = [x,y]$
- **output**: `z` = array of size (1,)

#### function`grad_f(indepvar)`:
- **description**: Computes the gradient of function f at coordinates (x, y), which corresponds to computing the partial derivate with respect to x and y $ \left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right) $
- **input** : `indepvar` = array of size (2,): $\boldsymbol{x} = [x,y]$
- **output**: `grad` = array of size (2,): $\nabla f(\boldsymbol{x}_k) = \left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right)\Big|_{\boldsymbol{x}_k} $

In [ ]:
# Define the function to be minimised (f(x,y) = 20(sin(x) * cos(y)) + x^2 + y^2) :
def f(indepvar):
    x = ...; y = ... # Retrieve the values x and y from indepvar
    z = ... # Use the function and the values x and y
    return z

# Define the partial derivatives of the function (f(x,y) = 20(sin(x) * cos(y)) + x^2 + y^2) with respect to x and y :
def grad_f(indepvar):
    grad = np.zeros((2,)) # Initialise the grad array
    x = ...; y = ... # Retrieve the values x and y from indepvar
    grad[0] = ... # df_dx
    grad[1] = ... # df_dy
    return grad

#### Gradient descent function
The goal is to write here the following steps of the gradient descent in the function below.

Steps:
- Calculate the function value $\boldsymbol{z}_k = f(\boldsymbol{x}_k)$
- Calculate next point as $\boldsymbol{x}_{k+1} = \boldsymbol{x}_k - \delta \cdot \nabla f(\boldsymbol{x}_k)$ (gradient descent step)
- Check convergence :  $|f(\boldsymbol{x}_{k+1} - f(\boldsymbol{x}_k)|< \epsilon$
- Save the solution : `history`

In [ ]:
# Define the gradient descent algorithm
def gradient_descent(start_indepvar, learning_rate, num_iterations):

    # Initialise values
    indepvar0 = ...
    z0 = ...
    
    # Parameters
    epsilon = 0.001
    history = ... # Initialise the history with zeros (size = (num_iterations, 3))
    
    # Perform the gradient descent iterations
    for i in range(num_iterations):
        # Calculate the gradients
        grad = ...
        
        # Update values
        indepvar = ...
        z        = ...
        
        # Save the history of the parameters
        history[i,:] = ...
        
        # Check convergence 
        if (abs(z-z0) < epsilon):
            # Reshape history and break
            history = history[:i, :]
            break
        else:
            # Reinitialise state 0:
            indepvar0 = ...; z0 = ...
    print(f'Last iteration n= {i+1}; Minimal function value = {z}; abs(z-z0)= {abs(z-z0)}')
    return indepvar, z, history

Execute the gradient descent function and see what happens.

1) What do you observe?

2) Afterwards, re-execute the code with other starting points (for example [5, 4]).<br>
Can you explain what you observe?

In [ ]:
# Perform gradient descent and plot the results
start_xy = [3, 3]
learning_rate = 0.01
num_iterations = 100
indepvar_opt, f_opt, history = ...

# Create Plots
fig= plt.figure(figsize=(10,5))

# Subplot 1: 3D Parametric Plot
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.grid()
surf = ax.plot_surface(X, Y, Z, cmap=plt.get_cmap('viridis'),alpha= 0.5 )
ax.set_title('3D Parametric Plot')
ax.scatter(*zip(*history), c='k', marker='o', s=5)
ax.scatter(*history[0], c='r', marker='o')
ax.scatter(*history[len(history)-1], c='b', marker='o')

ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)
ax.set_zlabel('f', labelpad=20)

# Subplot 2: 2D Contour Plot

ax = fig.add_subplot(1, 2, 2)
surf = ax.contourf(X, Y, Z, cmap=plt.get_cmap('viridis') )
ax.set_title('2D Contour Plot')
fig.colorbar(surf, shrink=0.5, aspect=8)

nit= history.shape[0]

for it in range(nit-1):
    plt.scatter(history[it,0], history[it,1], c='k', alpha=0.2)

plt.scatter(history[0,0], history[0,1] , c='r') # init
plt.scatter(history[nit-1,0], history[nit-1,1] , c='b') # end

# Set axes label
ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)

# # Set axes limit
# ax.set_xlim([0, 4])
# ax.set_ylim([2.5,3.5])

plt.show()

Visualise the evolution of the z-value over the iterations of the gradient descent algorithm

In [ ]:
# Create a plot that visualizes the evolution of the error over the iterations

z_start = f(start_xy)
z_totalHistory = np.concatenate(([z_start], history[:, 2]))

plt.plot(z_totalHistory, color = "black", linewidth = 5, zorder=1)
plt.axhline(y = f_opt, color = 'blue', linestyle='--', linewidth = 2, zorder=2)
plt.scatter(0, z_start, color = "red", s=60, zorder=3)

plt.annotate('Starting Point', (0, z_start), textcoords="offset points", xytext=(10, 0), ha='left', color='red')
plt.annotate('Optimal z-value', (0, f_opt), textcoords="offset points", xytext=(0, 10), ha='left', color='blue')

plt.xlabel("Iterations", fontsize = 15)
plt.ylabel("Function value", fontsize = 15)

plt.title("Evolution of the function value over the iterations")
plt.show()

### Gradient descent  estimated:

We not always know the analytical function of the gradient of the function.\
We can then estimate the gradient by computing the slope between two points close to each other.

We will create a function called `gradient_descent_estimated` that estimates the gradient.
 - To do:
     - Create the function similarly as before
     - Calculate the difference between the estimated gradient and the exact one at the last iteration

In [ ]:
# Define the gradient descent algorithm with estimation of the gradient
def gradient_descent_estimated(start_indepvar, learning_rate, num_iterations):

    # Initialise values
    indepvar0 = ...
    z0 = ...
    
    # Parameters
    epsilon = 0.001
    history = ...
    
    # Perform the gradient descent iterations
    for i in range(num_iterations):

        # Calculate the estimated gradients by computing the linear slope in the x and y direction in the current point
        dx = 0.01; grad_x = ...
        dy = 0.01; grad_y = ...
        
        grad = ... # combine grad_x and grad_y
        
        # Update values
        indepvar = ...
        z = ...
        
        # Save the history of the parameters
        history[i,:] = ...
        
        # Check convergence 
        if (abs(z-z0) < epsilon):
            # Reshape history and break
            history = history[:i, :]
            break
        else:
            # Reinitialize state 0:
            indepvar0 = ...; z0 = ...
    
    print(f'Last iteration n = {i+1}; Minimal function value = {z}; abs(z-z0) = {abs(z-z0)}')
    print(f'diff in grad = {grad - grad_f(indepvar)}')
    
    return indepvar, z, history

Apply now the estimated gradient descent algorithm

In [ ]:
# Perform gradient descent and plot the results
start_xy = [3, 3]
learning_rate = 0.01
num_iterations = 1000
indepvar_opt, f_opt, history_est = ... # Use the estimated gradient descent function that you just implemented

# Create Plots
fig= plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.grid()
surf = ax.plot_surface(X, Y, Z, cmap=plt.get_cmap('viridis'),alpha= 0.5 )
ax.set_title('3D Parametric Plot')
ax.scatter(*zip(*history_est), c='k', marker='o', s=5)
ax.scatter(*history_est[0], c='r', marker='o')
ax.scatter(*history_est[len(history_est)-1], c='b', marker='o')

# Set axes label
ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)
ax.set_zlabel('f', labelpad=20)

ax = fig.add_subplot(1, 2, 2)
surf = ax.contourf(X, Y, Z, cmap=plt.get_cmap('viridis') )
ax.set_title('2D Contour Plot')
fig.colorbar(surf, shrink=0.5, aspect=8)

nit= history_est.shape[0]

for it in range(nit-1):
    plt.scatter(history_est[it,0], history_est[it,1], c='k', alpha=0.2)

plt.scatter(history_est[0,0], history_est[0,1] , c='r') # init
plt.scatter(history_est[nit-1,0], history_est[nit-1,1] , c='b') # end

# Set axes label
ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)

# Set axes limit
ax.set_xlim([0, 4])
ax.set_ylim([2.5,3.5])

plt.show()

### Gradient descent using pytorch:

Now that we’ve understood the principle behind the gradient descent method, we can make direct use of the PyTorch library, which allows us to generalise what we’ve seen above.

First, install and import the PyTorch library

In [ ]:
# !pip install torch

import torch

Torch has its own way of handling variables and therefore differs from NumPy... So we need to change the way we define our variables, compute the function value and the gradient, and update the parameters. We will see how to do that in the code below.

In [ ]:
# Redefine the function to be minimised using torch instead of numpy (f(x,y) = 20(sin(x) * cos(y)) + x^2 + y^2) :
def f_torch(indepvar):
    x = indepvar[0]; y = indepvar[1] # Retrieve the values x and y from indepvar
    z = 20 * (torch.sin(x) * torch.cos(y)) + x**2 + y**2 # Use the function and the values x and y
    return z

In [ ]:
# Perform gradient descent using torch's autograd to compute the gradients
def gradient_descent_torch(start_indepvar, learning_rate, num_iterations):

    # Initialise values
    indepvar0 = torch.tensor(start_indepvar, requires_grad=True)
    z0 = f_torch(indepvar0).detach() # Detach from the computation graph

    # Parameters
    epsilon = 0.001
    history = torch.zeros((num_iterations, 3))

    # Perform the gradient descent iterations
    for i in range(num_iterations):

        # Calculate the gradients
        z = f_torch(indepvar0) # Evaluate the function
        z.backward() # Compute the gradients
        grad = indepvar0.grad # Retrieve the gradients

        # Update values
        indepvar = indepvar0 - learning_rate * grad
        z = f_torch(indepvar).detach() # Detach from the computation graph

        # Save the history of the function
        history[i, :2] = indepvar.detach() # Save the parameters
        history[i, 2] = z.detach() # Save the function value

        # Check convergence
        if (abs(z - z0) < epsilon):
            # Reshape history and break
            history = history[:i, :]
            break
        else:
            # Create a copy of the current values to be able to recompute the gradient on it on the next iteration
            indepvar0 = indepvar.detach().requires_grad_(True); z0 = z.detach()
    
    print(f'Last iteration n = {i+1}; Minimal function value = {z}; abs(z-z0) = {abs(z-z0)}')
    
    return indepvar, z, history

In [ ]:
# Perform gradient descent and plot the results
start_xy = [3., 3.]
learning_rate = 0.01
num_iterations = 1000
indepvar_opt, f_opt, history_est = ... # Use the estimated gradient descent function that you just implemented

# Create Plots
fig= plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.grid()
surf = ax.plot_surface(X, Y, Z, cmap=plt.get_cmap('viridis'),alpha= 0.5 )
ax.set_title('3D Parametric Plot')
ax.scatter(*zip(*history_est), c='k', marker='o', s=5)
ax.scatter(*history_est[0], c='r', marker='o')
ax.scatter(*history_est[len(history_est)-1], c='b', marker='o')

# Set axes label
ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)
ax.set_zlabel('f', labelpad=20)

ax = fig.add_subplot(1, 2, 2)
surf = ax.contourf(X, Y, Z, cmap=plt.get_cmap('viridis') )
ax.set_title('2D Contour Plot')
fig.colorbar(surf, shrink=0.5, aspect=8)

nit = history_est.shape[0]

for it in range(nit-1):
    plt.scatter(history_est[it,0], history_est[it,1], c='k', alpha=0.2)

plt.scatter(history_est[0,0], history_est[0,1] , c='r') # init
plt.scatter(history_est[nit-1,0], history_est[nit-1,1] , c='b') # end

# Set axes label
ax.set_xlabel('x', labelpad=20)
ax.set_ylabel('y', labelpad=20)

# Set axes limit
ax.set_xlim([0, 4])
ax.set_ylim([2.5,3.5])

plt.show()

Below is a summary of the three different implementations of the gradient descent method described above.

| Approach | How it works | When to use |
|---|---|---|
| **Analytical gradient** | Derive $\nabla_\theta \mathcal{L}$ by hand and hard-code it | Rarely done in practice — tedious and error-prone for complex models |
| **Finite differences** | Approximate $\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \Delta\theta) - \mathcal{L}(\theta)}{\Delta\theta}$ | Useful to verify analytical gradients, but too slow for large models |
| **Automatic differentiation** | The library (e.g. PyTorch) tracks operations and computes gradients exactly | The standard in practice — efficient, exact, and works for any model |

Note that there are many other optimisation methods that are much simpler to apply, but these are not covered in the session (see: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fmin.html).

## Exercise 3: Back to our non-linear regression problem

Apply gradient descent method to find the optimal coefficients.

In [ ]:
# Fix the seed for reproducibility
np.random.seed(42)

# We collected n=100 measures
n = 100
x = np.linspace(0, 10, n) # 10 seconds experiment

# Let's imagine we know the coefficients
A = 10
alpha = 0.5
omega = 2
phi = 0.5

# We made n noisy observations
msrmt_error = 0.5
y_obs = A * np.sin(omega * x + phi) * np.exp(-alpha * x) + np.random.normal(0, msrmt_error, size=x.shape)

# Plot the data
plt.scatter(x, y_obs, color='k')
plt.show()

The function, the error metric and the error function are also implemented again in PyTorch.

In [ ]:
# Redefine the function to be optimised using torch instead of numpy (f(x,y) = A·exp(-alpha·x)·sin(w·x + phi)) :
def f_torch(x, indepvar):
    y = ...
    return y

# Compute the relative error between the true and predicted value
def rmse_torch(y_true, y_pred):
    return ...

# Given the input x, the output y and the weights, return the error
def error_torch(x, y, indepvar):
    y_pred = ...
    error = ...
    return error

Given the functions above, write the gradient descent algorithm like in the previous exercise.

Here, the gradient will be computed with respect to the parameters we want to determine and the goal is to minimise the error between the true y-values and the predicted y-values of our regression model.

In [ ]:
nbr_it = 1000 # Number of iterations
learning_rate = 0.1 # Learning rate

# Fix the seed for reproducibility
np.random.seed(42)
Params = torch.randn((4,), requires_grad=True)

history_error = torch.zeros(nbr_it)

x_torch = torch.tensor(x)
y_torch = torch.tensor(y_obs)

for i in range(nbr_it):

    # Compute the predicted y-values given the current beta and x-values
    y_pred = ...

    # Compute the error
    error = ...
    ...

    # Retrieve the gradient value
    gradient = ...

    # Update the weights Beta given the gradient and the learning rate
    Params = ...

    # Create a copy of the current Beta value to be able to recompute the gradient on it on the next iteration
    Params = ...

    # Store the history of the errors to plot it afterwards
    history_error[i] = error.detach()

print(f'Optimal parameters = {Params.detach().numpy()}')
print(f'True parameters = {A}, {alpha}, {omega}, {phi}')
print(f'Error = {error_torch(x_torch, y_torch, Params)}')

In [ ]:
# We can plot the dataset, the original non-linear function and the reconstructed
plt.scatter(x_torch, y_torch, c='k')
plt.plot(x_torch, y_pred.detach().numpy(), c='r')
plt.show()